# Migration Phase 4: `FeatureStore` for the new tabular-derivatives schema

`src/neuroalign/data/preprocessing/feature_store.py` is rewritten for the
`TabularDerivativesLoader` long-format outputs (Phases 2-3):

- **Long format**: `anatomical.parquet` (Schaefer2018N400n7 + Tian2020S2,
  cortex+subcortex concatenated) and `diffusion/<software>.parquet` (one
  file per software, e.g. `DSIStudio.parquet`, `AMICONODDI.parquet`).
- **`ANATOMICAL_METRICS`** / **`DIFFUSION_METRICS`** updated to the new
  column sets (`thickness_mean_mm`, `gray_matter_volume_mm3`, ...,
  `mean`, `robust_mean`, `percentile_5`, ...).
- **Wide format**: `anat_<metric>.parquet` (one per anatomical metric) and
  `<software>_<model>_<param>[_<desc>]_<metric>.parquet` (one per diffusion
  combination/metric), pivoted on `label` (region).
- **`save_tiv()`** now just extracts `tiv_mm3` straight from the anatomical
  long format - no MATLAB/CAT12 step.
- All join keys are now `uid` + `session_id` (`META_COLS`), matching
  `BehavioralLoader` and `TabularDerivativesLoader`.

This notebook builds a small demo store under `data/processed/` (gitignored)
from a 5-session sample.

In [1]:
from pathlib import Path
from dotenv import load_dotenv
import os

load_dotenv(Path.cwd().parent / ".env")

from neuroalign.data.loaders import BehavioralLoader, TabularDerivativesLoader
from neuroalign.data.preprocessing.feature_store import FeatureStore

beh = BehavioralLoader(os.environ["BRAINLINK_DB_PATH"])
loader = TabularDerivativesLoader(
    os.environ["TABULAR_DERIVATIVES_ROOT"],
    atlas_name=os.environ["ATLAS_NAME"],
    anat_atlases=tuple(os.environ["ANAT_ATLASES"].split(",")),
    session_variant=os.environ["SESSION_VARIANT"],
)

store_dir = Path.cwd().parent / "data" / "processed" / "phase4_demo"
store = FeatureStore(store_dir)
print(f"store_dir={store_dir}")

store_dir=/home/galkepler/Projects/neuroalign/data/processed/phase4_demo


## 1. Load a small sample

Reuse the 5-session sample with diffusion data from Phase 3, plus the
matching anatomical data.

In [2]:
import pandas as pd

sessions = beh.get_sessions()
tabular_root = Path(os.environ["TABULAR_DERIVATIVES_ROOT"])
atlas = os.environ["ATLAS_NAME"]

found = []
for uid, sid in sessions[["uid", "session_id"]].drop_duplicates().itertuples(index=False):
    if (tabular_root / f"sub-{uid}" / f"ses-{sid}" / "dwi" / f"atlas-{atlas}").exists():
        found.append((uid, sid))
        if len(found) >= 5:
            break

sample = pd.DataFrame(found, columns=["uid", "session_id"])
sample_sessions = sessions.merge(sample, on=["uid", "session_id"])

anat = loader.load_anatomical(sample)
dwi = loader.load_diffusion(sample)
print(f"anat: {anat.shape}, dwi: {dwi.shape}, sessions: {sample_sessions.shape}")

anat: (864, 31), dwi: (123120, 33), sessions: (5, 17)


## 2. Save long format

`save_anatomical_long()` writes one `anatomical.parquet`.
`save_diffusion_long()` splits by `software` into `diffusion/<software>.parquet`.

In [3]:
anat_name = store.save_anatomical_long(anat, atlas_name=atlas)
dwi_names = store.save_diffusion_long(dwi, atlas_name=atlas)
print("anat:", anat_name)
print("dwi:", dwi_names)
print()
print("long formats:", store.list_long_formats())

anat: anatomical
dwi: ['diffusion_AMICONODDI', 'diffusion_DIPYDKI', 'diffusion_DIPYMAPMRI', 'diffusion_DSIStudio', 'diffusion_MRtrix3actHSVS']

long formats: ['anatomical', 'diffusion_AMICONODDI', 'diffusion_DIPYDKI', 'diffusion_DIPYMAPMRI', 'diffusion_DSIStudio', 'diffusion_MRtrix3actHSVS']


## 3. TIV

`save_tiv()` extracts `tiv_mm3` directly from the anatomical long format -
one row per (uid, session_id).

In [4]:
tiv_name = store.save_tiv(anat)
store.load_tiv()

,uid,session_id,tiv_mm3
0,S629697,202601111959,1.415257e+06
1,S076379,202410131245,1.500654e+06


## 4. Metadata

`save_metadata()` stores the `BehavioralLoader` session table directly
(`uid`, `session_id`, `AGE`, `sex`, `lab`, ...).

In [5]:
store.save_metadata(sample_sessions)
store.load_metadata()[["uid", "session_id", "lab", "AGE", "sex"]]

,uid,session_id,lab,AGE,sex
0,S629697,202601111959,YA,37.691992,Female
1,S667793,202601181721,YA,27.641342,Female
2,S307570,202601051400,SNBB,25.262149,Female
3,S076379,202410131245,TS,22.260000,Female
4,S105498,202601141203,YA,36.720055,Female


## 5. Generate wide features

One `anat_<metric>.parquet` per anatomical metric, one
`<software>_<model>_<param>[_<desc>]_<metric>.parquet` per diffusion
combination/metric.

In [6]:
generated = store.generate_wide_features()
print(f"{len(generated)} wide features generated")
print("anatomical:", store.list_features("anatomical")[:5], "...")
print("diffusion:", store.list_features("diffusion")[:5], "...")

1277 wide features generated
anatomical: ['anat_brain_seg_no_vent_mm3', 'anat_brain_seg_vol_mm3', 'anat_cortex_vol_mm3', 'anat_curvature_index', 'anat_folding_index'] ...
diffusion: ['AMICONODDI_noddi_direction_coverage', 'AMICONODDI_noddi_direction_cv', 'AMICONODDI_noddi_direction_excess_kurtosis', 'AMICONODDI_noddi_direction_iqr_filtered_mean', 'AMICONODDI_noddi_direction_iqr_filtered_std'] ...


## 6. Load a feature

`load_feature()` merges the wide feature with metadata (`AGE`, `sex`) and
TIV on `uid` + `session_id`.

In [7]:
ct = store.load_feature("anat_thickness_mean_mm", include_tiv=True)
print(ct.shape)
ct[["uid", "session_id", "AGE", "sex", "tiv_mm3"] + [c for c in ct.columns if c.startswith("LH_Vis")][:3]]

(2, 418)


,uid,session_id,AGE,sex,tiv_mm3
0,S076379,202410131245,22.260000,Female,1.500654e+06
1,S629697,202601111959,37.691992,Female,1.415257e+06


In [8]:
fa = store.load_feature("DSIStudio_tensor_fa_mean")
print(fa.shape)
fa[["uid", "session_id", "AGE"] + [c for c in fa.columns if c.startswith("LH_Vis")][:3]]

(5, 449)


,uid,session_id,AGE,LH_Vis_1,LH_Vis_10,LH_Vis_11
0,S076379,202410131245,22.260000,0.161196,0.196716,0.189170
1,S105498,202601141203,36.720055,0.197376,0.209543,0.201873
2,S307570,202601051400,25.262149,0.207833,0.201729,0.190115
3,S629697,202601111959,37.691992,0.192400,0.212407,0.190610
4,S667793,202601181721,27.641342,0.171997,0.173567,0.182842


## 7. Store summary

In [9]:
store.summary()

{'root_dir': '/home/galkepler/Projects/neuroalign/data/processed/phase4_demo',
 'atlas_name': 'Schaefer2018N400n7Tian2020S2',
 'n_sessions': 5,
 'n_subjects': 5,
 'long_formats': ['anatomical',
  'diffusion_AMICONODDI',
  'diffusion_DIPYDKI',
  'diffusion_DIPYMAPMRI',
  'diffusion_DSIStudio',
  'diffusion_MRtrix3actHSVS'],
 'n_wide_features': 1278,
 'anatomical_features': ['tiv',
  'anat_num_vertices',
  'anat_surface_area_mm2',
  'anat_gray_matter_volume_mm3',
  'anat_thickness_mean_mm',
  'anat_thickness_std_mm',
  'anat_mean_curvature',
  'anat_gaussian_curvature',
  'anat_folding_index',
  'anat_curvature_index',
  'anat_white_surf_area_mm2',
  'anat_brain_seg_vol_mm3',
  'anat_brain_seg_no_vent_mm3',
  'anat_cortex_vol_mm3',
  'anat_supratentorial_vol_mm3',
  'anat_num_voxels',
  'anat_volume_mm3',
  'anat_intensity_mean',
  'anat_intensity_std',
  'anat_intensity_min',
  'anat_intensity_max',
  'anat_intensity_range',
  'anat_intensity_snr',
  'anat_subcort_gray_mm3'],
 'diffusio